**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/06-vit-from-paper.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/vit-figure-1) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7F · Vision Transformer from the Paper (Dosovitskiy et al., 2021)

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 75 min (training ≈ 1 min on CPU) | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | patches → tokens, the patch embedding, the [class] token and position embeddings, ViT-B/16 with exactly 86,567,656 parameters, and a tiny ViT that reads handwritten digits |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

📄 **Paper:** A. Dosovitskiy et al., *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*,
ICLR 2021 · [arXiv:2010.11929](https://arxiv.org/abs/2010.11929) · [PDF](https://arxiv.org/pdf/2010.11929) · read
**§3.1, Figure 1, Equations 1–4 and Table 1**.

> *"We split an image into fixed-size patches, linearly embed each of them, add position embeddings, and feed the
> resulting sequence of vectors to a standard Transformer encoder. In order to perform classification, we use the
> standard approach of adding an extra learnable "classification token" to the sequence."* (Figure 1 caption)

The four equations of §3.1, in plain words:

| Eq. | Formula | In plain words |
|---|---|---|
| 1 | $z_0 = [x_{class};\, x_p^1E;\, \dots;\, x_p^NE] + E_{pos}$ | cut into N patches, flatten, multiply by E; put a learned [class] token in front; add learned positions |
| 2 | $z'_\ell = \mathrm{MSA}(\mathrm{LN}(z_{\ell-1})) + z_{\ell-1}$ | attention sub-layer, **LayerNorm first** ("pre-norm"), then residual |
| 3 | $z_\ell = \mathrm{MLP}(\mathrm{LN}(z'_\ell)) + z'_\ell$ | MLP sub-layer (two linear layers with GELU), pre-norm, residual |
| 4 | $y = \mathrm{LN}(z_L^0)$ | the final [class] token, normalised, is the image representation |

Everything else is the Transformer encoder you built in Lab 7C. **Reading tip:** this paper reuses another paper, so
its method section is only 1½ pages. When a paper says "we follow [X]", read X first.

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## 1 · An image is a sequence of patches

With $P$ = patch size, an $H \times W \times C$ image becomes $N = HW/P^2$ patches, each flattened to $P^2 C$ numbers.
ViT-B/16 on a 224 × 224 RGB image: N = 224²/16² = **196** patches of 16·16·3 = **768** numbers each.

### 🖐 Your turn 1: `patchify(x, P)`: (B, C, H, W) → (B, N, P·P·C), patches in reading order (row by row)
<details><summary><b>Hint 1</b></summary>

Split the two image axes: `x.reshape(B, C, H // P, P, W // P, P)` gives axes (B, C, rows, P, cols, P).

</details>
<details><summary><b>Hint 2</b></summary>

Bring the patch-index axes forward: `.permute(0, 2, 4, 3, 5, 1)` → (B, rows, cols, P, P, C).

</details>
<details><summary><b>Hint 3</b></summary>

Then `.reshape(B, N, P * P * C)`. A plain `x.reshape(B, N, -1)` without the permute gives stripes, not squares!

</details>

### 📄 ↔ 💻 Paper ↔ Code: §3.1: reshape the image into flattened patches → `patchify`

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, §3.1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_31.png" width="100%" alt="pc_v_31.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def patchify(x, P):        # x (B, C, H, W) -&gt; x_p (B, N, P*P*C), N = H*W / P^2
    # 1. split H into (H/P, P) and W into (W/P, P)
    # 2. move the patch grid first and the pixels + channels last (permute)
    # 3. flatten every patch: (B, N, P*P*C), patches in reading order</pre>
<b>Shapes:</b> (B, 3, 224, 224) → (B, 196, 768): N = 224² / 16² = 196 patches, each 16·16·3 = 768 numbers
</td></tr>
</table>

In [ ]:
def patchify(x, P):
    B, C, H, W = x.shape
    # TODO
    return x

In [ ]:
C.check_patchify(patchify)

In [ ]:
from sklearn.datasets import load_sample_image
photo = torch.from_numpy(load_sample_image("china.jpg")[:416, 100:516]).permute(2, 0, 1).float() / 255   # (3, 416, 416)
photo = F.interpolate(photo[None], size=(224, 224), mode="bilinear", align_corners=False)              # (1, 3, 224, 224)
p = patchify(photo, 16)
print("image", tuple(photo.shape), "→ patches", tuple(p.shape))
plt.figure(figsize=(4, 4)); plt.imshow(photo[0].permute(1, 2, 0)); plt.title("224 × 224 image = 14 × 14 patches"); plt.axis("off")
for k in range(0, 225, 16):
    plt.axhline(k - 0.5, c="w", lw=0.6); plt.axvline(k - 0.5, c="w", lw=0.6)
plt.show()
fig, ax = plt.subplots(1, 14, figsize=(12, 1.3))      # row 9 of the grid (patches 126 … 139), as a sequence of "words"
for j, i in enumerate(range(9 * 14, 10 * 14)):
    ax[j].imshow(p[0, i].reshape(16, 16, 3)); ax[j].set_title(str(i), fontsize=8); ax[j].axis("off")
fig.suptitle("14 of the 196 'words' of this image: patches 126 … 139, row 9 of the grid (each = 16·16·3 = 768 numbers)", fontsize=10, y=1.15)
plt.show()

## 2 · Patch embedding: "linear projection of flattened patches" = a strided convolution

Eq. 1 multiplies every flattened patch by a matrix $E$ of size $(P^2C) \times D$. A `Conv2d` with kernel $P$ and stride
$P$ does **exactly** the same (each output position sees one patch, no overlap) and is faster. Let's prove it:

### 📄 ↔ 💻 Paper ↔ Code: Figure 1 and Eq. (1): "linear projection of flattened patches" → `PatchEmbedding`

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, §3.1, Figure 1, Eq. (1)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_vit_fig1.png" width="100%" alt="p_vit_fig1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_eq.png" width="100%" alt="pc_v_eq.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class PatchEmbedding(nn.Module):
    # Eq. 1:  x_p^i E,  E in R^{(P^2 C) x D}   (the same E for every patch)
    # = ONE nn.Conv2d(in_ch, dim, kernel_size=P, stride=P):
    #   each P x P patch is seen exactly once by the same weights
    def forward(self, x):    # (B, C, H, W) -&gt; (B, D, H/P, W/P) -&gt; (B, N, D)</pre>
</td></tr>
</table>

In [ ]:
P, D = 16, 8
conv = nn.Conv2d(3, D, kernel_size=P, stride=P)
lin = nn.Linear(3 * P * P, D)
with torch.no_grad():                                 # copy the conv kernel into the linear layer (same ordering as patchify)
    lin.weight.copy_(conv.weight.permute(0, 2, 3, 1).reshape(D, -1)); lin.bias.copy_(conv.bias)
a = conv(photo).flatten(2).transpose(1, 2)            # (1, 196, D)
b = lin(patchify(photo, P))                           # (1, 196, D)
print("conv:", tuple(a.shape), "· linear on patches:", tuple(b.shape), "· same numbers:", torch.allclose(a, b, atol=1e-5))

### 🖐 Your turn 2: `PatchEmbedding(img_size, patch, in_ch, dim)` → (B, N, D)
<details><summary><b>Hint 1</b></summary>

`self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)`

</details>
<details><summary><b>Hint 2</b></summary>

forward: `self.proj(x)` is (B, D, H/P, W/P) → `.flatten(2).transpose(1, 2)` → (B, N, D) (Lab 7A, `to_tokens`!).

</details>

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch, in_ch, dim):
        super().__init__()
        self.n_patches = (img_size // patch) ** 2
        self.proj = None   # TODO

    def forward(self, x):
        # TODO
        return x

In [ ]:
C.check_patch_embed(PatchEmbedding)

## 3 · The encoder block (Eq. 2–3) and the whole ViT (Eq. 1, 4)

The encoder block is given (we use PyTorch's `nn.MultiheadAttention`, which you re-built in Lab 7B). Note the
**pre-norm** order: LayerNorm *inside* the residual branch, unlike the 2017 Transformer.

### 🖐 Your turn 3: `ViT.forward`
1. patch embedding → (B, N, D)
2. put the [class] token in front: `cls_token` is (1, 1, D) → expand to (B, 1, D) → `torch.cat` → (B, N+1, D)
3. add `pos_embed` (1, N+1, D), dropout
4. the encoder blocks, the final LayerNorm, and the head on token 0 only
<details><summary><b>Hint 1</b></summary>

`cls = self.cls_token.expand(x.size(0), -1, -1)`

</details>
<details><summary><b>Hint 2</b></summary>

`x = torch.cat([cls, x], dim=1) + self.pos_embed`

</details>
<details><summary><b>Hint 3</b></summary>

`return self.head(self.norm(self.blocks(self.drop(x)))[:, 0])`

</details>

### 📄 ↔ 💻 Paper ↔ Code: Eq. (1)–(4) → `ViT.forward`

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, §3.1, Eq. (1)–(4)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_eq.png" width="100%" alt="pc_v_eq.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_vit_fig1.png" width="100%" alt="p_vit_fig1.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def forward(self, x):
    # Eq. 1: z0 = [x_class; x_p E] + E_pos
    #   patch_embed(x)                        (B, N, D)
    #   cls_token.expand(B, -1, -1), cat on dim 1 -&gt; (B, N+1, D)
    #   + pos_embed  (1, N+1, D) broadcasts over the batch
    # Eq. 2-3: z = blocks(z)    L x EncoderBlock (pre-norm, given above)
    # Eq. 4:   y = LN(z_L^0)    norm(z), then token 0 only: [:, 0]
    # head:    Linear(D, n_classes)            (B, n_classes)</pre>
</td></tr>
</table>

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(mlp_dim, dim), nn.Dropout(dropout))
        self.attn_weights = None

    def forward(self, x):
        h = self.norm1(x)
        a, self.attn_weights = self.attn(h, h, h, need_weights=True, average_attn_weights=True)
        x = x + a                         # Eq. 2
        x = x + self.mlp(self.norm2(x))   # Eq. 3
        return x

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch=16, in_ch=3, n_classes=1000, dim=768, depth=12, heads=12, mlp_dim=3072, dropout=0.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch, in_ch, dim)
        n = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[EncoderBlock(dim, heads, mlp_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        # TODO: Eq. 1 → blocks (Eq. 2–3) → Eq. 4 → head
        return x

In [ ]:
vit_b16 = ViT()              # Table 1, "ViT-Base": L = 12, D = 768, MLP 3072, 12 heads, patch 16
C.check_vit_b16(vit_b16)
del vit_b16                  # free ~350 MB of memory

## 4 · Train a tiny ViT on handwritten digits (≈ 1 min on CPU)

ViT-B/16 needs hundreds of millions of images (the paper pre-trains on JFT-300M). We shrink everything: 8 × 8 digit
images, **2 × 2 patches → 16 tokens**, D = 64, 4 blocks, 4 heads (136 k parameters). Optimiser as in the paper: AdamW
with weight decay and a warm-up + decay schedule.

### 📄 ↔ 💻 Paper ↔ Code: Table 1: the model variants are just constructor arguments

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, §4.1, Table 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_table1.png" width="100%" alt="pc_v_table1.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># Table 1 row &quot;ViT-Base&quot; with patch size 16 = &quot;ViT-B/16&quot;:
vit_b16 = ViT(img_size=224, patch=16, in_ch=3, n_classes=1000,
              dim=768, depth=12, heads=12, mlp_dim=3072)
print(C.count_params(vit_b16))     # 86 567 656, Table 1 says 86M
&nbsp;
# our tiny ViT for 8 x 8 digits: the same code, smaller numbers
tiny = ViT(img_size=8, patch=2, in_ch=1, n_classes=10,
           dim=64, depth=4, heads=4, mlp_dim=128)</pre>
</td></tr>
</table>

In [ ]:
Xtr, ytr, Xte, yte = C.load_digits_tensors()

def train_vit(patch=2, use_pos=True, epochs=40, lr=2e-3):
    C.seed_everything(0)
    m = ViT(img_size=8, patch=patch, in_ch=1, n_classes=10, dim=64, depth=4, heads=4, mlp_dim=128, dropout=0.1).to(device)
    if not use_pos:                                  # ablation: no position information at all
        m.pos_embed.data.zero_(); m.pos_embed.requires_grad_(False)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.05)
    steps = epochs * math.ceil(len(Xtr) / 64)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 64):
            idx = perm[i:i + 64]
            loss = F.cross_entropy(m(Xtr[idx].to(device)), ytr[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    m.eval()
    with torch.no_grad():
        acc = (m(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
    return m, acc

t0 = time.time()
tiny, acc = train_vit()
print(f"tiny ViT: {C.count_params(tiny):,} parameters · test accuracy {acc:.1%} · {time.time() - t0:.0f} s")
C.check_vit_accuracy(acc)

## 5 · Where does the [class] token look?

The attention weights of the last block, row 0 (the [class] token) over the 16 patches, drawn on top of each digit.

In [ ]:
with torch.no_grad():
    tiny(Xte[:8].to(device))
att = tiny.blocks[-1].attn_weights[:, 0, 1:].reshape(-1, 4, 4).cpu()     # [class] → 16 patches, as a 4 × 4 grid
fig, ax = plt.subplots(2, 8, figsize=(12, 3.6))
for i in range(8):
    ax[0, i].imshow(Xte[i, 0], cmap="gray"); ax[0, i].set_title(f"label {yte[i].item()}"); ax[0, i].axis("off")
    ax[1, i].imshow(att[i], cmap="magma"); ax[1, i].axis("off")          # bright = the [class] token looks here
ax[1, 0].set_title("[class] → 4×4 patches", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 6 · Ablation: what do the position embeddings do? (≈ 1 min)

### 📄 ↔ 💻 Paper ↔ Code: the paper's results next to yours

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, Table 2, Figure 3</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_table2.png" width="100%" alt="pc_v_table2.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_fig3.png" width="100%" alt="pc_v_fig3.png"><br></td>
<td width="50%" valign="top">
<b>📄 The paper reports</b>
<ul>
<li>ViT-H/14 pre-trained on JFT-300M: <b>88.55 %</b> ImageNet top-1 (Table 2)</li>
<li>Figure 3: pre-trained only on ImageNet-1k, ViTs are <b>below</b> ResNets (BiT); they win only with ImageNet-21k or JFT-300M</li>
<li>Appendix D.4, Table 8: <b>no</b> position embedding 61.4 % vs learned 1-D 64.2 % (ViT-B/16, ImageNet 5-shot linear)</li>
</ul>
<b>💻 You measure in this notebook</b>
<ul>
<li>a tiny ViT on 8 × 8 digits, with and without position embeddings</li>
<li>try training on fewer images: the gap to a small CNN grows, which is Figure 3's lesson</li>
</ul>
</td></tr>
</table>

In [ ]:
_, acc_nopos = train_vit(use_pos=False)
_, acc_p4 = train_vit(patch=4)
print(f"{'tiny ViT, 2×2 patches (16 tokens)':38s} {acc:.1%}")
print(f"{'… without position embeddings':38s} {acc_nopos:.1%}")
print(f"{'… 4×4 patches (only 4 tokens)':38s} {acc_p4:.1%}")

**Think:** without positions the model sees a *bag* of patches: it knows which strokes exist but not where. Which digits
can it still tell apart, which not? (Compare with Lab 7C, where removing positions made reversing impossible.) The paper's
Appendix D.4 runs the same ablation on ImageNet.

## ✍️ Your turn
1. ViT-B/16 on a 384 × 384 image: how many tokens? Attention cost grows with N²: how many times more than at 224 × 224?
2. Where are most of ViT-B/16's 86.6 M parameters: patch embedding, attention, or MLPs? Compute it from the shapes (or with `C.count_params` on parts).
3. Change the tiny ViT to `depth=1`, then `depth=8`. Record parameters and accuracy.
4. ⭐ Load pretrained weights into **your** ViT-B/16: `torchvision.models.vit_b_16(weights="IMAGENET1K_V1")` uses different names (`conv_proj`, `class_token`, `encoder.layers.encoder_layer_0.ln_1` …). Write the name mapping and classify `china.jpg`. (This is how you port a model between codebases.)

**Next:** [Lab 7G · Bug hunt 🐞](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/07-paper-bug-hunt.ipynb)